# 00 — Raw Data Audit and Structural Preparation

This notebook is deliberately **not feature engineering**. It creates a reproducible, audited foundation for later state-of-the-art features.

It will:

1. locate the repository and raw Kaggle files;
2. load every CSV and create a version/hash manifest;
3. run integrity and leakage-risk checks;
4. standardize men's and women's tables;
5. orient games by the competition's lower-TeamID convention;
6. reshape detailed box scores to a team-perspective long table;
7. parse Stage 1 and Stage 2 matchup IDs;
8. save canonical Parquet tables under `data/interim/`.

It will **not** compute Elo, efficiencies, rolling form, strength of schedule, rankings, model matrices, or predictions.

In [ ]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import pandas as pd
import yaml

from march_mania.io import build_inventory, load_csv_tables, write_manifest
from march_mania.paths import get_project_paths
from march_mania.reshape import (
    canonicalize_compact_results,
    combine_game_cities,
    combine_seasons,
    combine_seeds,
    combine_team_conferences,
    combine_teams,
    detailed_results_to_team_long,
    parse_submission_tables,
)
from march_mania.validation import run_raw_data_audit

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

PATHS = get_project_paths()
CONFIG = yaml.safe_load((PATHS.root / "configs" / "base.yaml").read_text(encoding="utf-8"))

print("Project root:", PATHS.root)
print("Raw data:", PATHS.raw)
print("Python:", sys.executable)
print("Python version:", platform.python_version())
print("pandas:", pd.__version__)
assert "ml-modeling" in str(sys.executable).lower(), "Select the Python (ml-modeling) kernel."


## 1. Load every competition CSV

The raw directory must contain the extracted Kaggle files. Raw files are never modified.

In [ ]:
tables, csv_files = load_csv_tables(PATHS.raw)
print(f"Loaded {len(tables):,} CSV tables")
print("\n".join(sorted(tables)))


## 2. Freeze a manifest and inventory

SHA-256 hashes make the exact raw-data version reproducible. This matters because Kaggle updated the 2026 files during the competition.

In [ ]:
inventory = build_inventory(tables, csv_files)
inventory.to_csv(PATHS.reports / "table_inventory.csv", index=False)
write_manifest(inventory, PATHS.reports / "data_manifest.json")
inventory[["Table", "Rows", "Columns", "SizeMB"]]


## 3. Audit raw integrity and leakage risks

Warnings are not automatically deleted. They are surfaced for investigation. In particular, target-season tournament results must never enter a historical replay of a 2026 forecast.

In [ ]:
audit = run_raw_data_audit(tables, target_season=int(CONFIG["target_season"]))
audit.to_csv(PATHS.reports / "raw_data_audit.csv", index=False)

print(audit.groupby(["Severity", "Passed"], dropna=False).size())
audit.loc[~audit["Passed"]].reset_index(drop=True)


In [ ]:
errors = audit.loc[(audit["Severity"] == "ERROR") & (~audit["Passed"])]
if not errors.empty:
    raise AssertionError(
        "Blocking raw-data audit failures were found. Review reports/data_quality/raw_data_audit.csv"
    )
print("No blocking raw-data audit failures.")


## 4. Canonical reference tables

These operations add gender/source metadata and parse structural identifiers. They do not aggregate predictive features.

In [ ]:
teams = combine_teams(tables)
seasons = combine_seasons(tables)
seeds = combine_seeds(tables)
team_conferences = combine_team_conferences(tables)
game_cities = combine_game_cities(tables)

print("teams:", teams.shape)
print("seasons:", seasons.shape)
print("seeds:", seeds.shape)
print("team_conferences:", team_conferences.shape)
print("game_cities:", game_cities.shape)


## 5. Canonical compact game table

Every game is rewritten so `Team1ID < Team2ID`, exactly matching the Kaggle submission convention. `Team1Win` is the historical target. Regular-season and NCAA games remain explicitly separated.

In [ ]:
games_compact = canonicalize_compact_results(tables)
tournament_targets = games_compact.loc[games_compact["GameType"] == "NCAA"].copy()

assert (games_compact["Team1ID"] < games_compact["Team2ID"]).all()
assert games_compact["GameKey"].is_unique

print("All compact games:", games_compact.shape)
print("Tournament target rows:", tournament_targets.shape)
games_compact.head()


## 6. Detailed box scores in team-perspective long form

Each detailed game becomes exactly two rows: one for each team. Raw box-score fields are renamed consistently as `Team...` and `Opp...`. No efficiencies or rolling aggregates are calculated yet.

In [ ]:
games_detailed_team_long = detailed_results_to_team_long(tables)
rows_per_game = games_detailed_team_long.groupby("GameKey").size()
assert rows_per_game.eq(2).all()

print("Detailed team-game rows:", games_detailed_team_long.shape)
print("Unique detailed games:", games_detailed_team_long["GameKey"].nunique())
games_detailed_team_long.head()


## 7. Parse every required submission matchup

In [ ]:
submission_matchups = parse_submission_tables(tables)
assert (submission_matchups["Team1ID"] < submission_matchups["Team2ID"]).all()
assert submission_matchups["Gender"].isin(["M", "W"]).all()

submission_matchups.groupby(["SubmissionFile", "Season", "Gender"]).size().to_frame("Rows")


## 8. Preserve optional competition tables

Massey rankings are men's-only. Coaches, cities, conference tournaments, secondary tournaments, and bracket slots remain separate reference tables so later work can use them without re-reading raw CSVs.

In [ ]:
optional_outputs = {
    "massey_ordinals_men": tables.get("MMasseyOrdinals"),
    "coaches_men": tables.get("MTeamCoaches"),
    "cities": tables.get("Cities"),
    "conferences": tables.get("Conferences"),
    "men_tourney_slots": tables.get("MNCAATourneySlots"),
    "women_tourney_slots": tables.get("WNCAATourneySlots"),
    "men_seed_round_slots": tables.get("MNCAATourneySeedRoundSlots"),
}
{k: None if v is None else v.shape for k, v in optional_outputs.items()}


## 9. Write interim Parquet tables

Parquet preserves types, loads quickly, and is substantially better than repeatedly parsing CSVs. The files are generated artifacts and remain excluded from Git.

In [ ]:
outputs = {
    "teams": teams,
    "seasons": seasons,
    "seeds": seeds,
    "team_conferences": team_conferences,
    "game_cities": game_cities,
    "games_compact_canonical": games_compact,
    "tournament_targets": tournament_targets,
    "games_detailed_team_long": games_detailed_team_long,
    "submission_matchups": submission_matchups,
}
outputs.update({name: frame for name, frame in optional_outputs.items() if frame is not None})

written = []
for name, frame in outputs.items():
    path = PATHS.interim / f"{name}.parquet"
    frame.to_parquet(path, index=False, compression="zstd")
    written.append({"Table": name, "Rows": len(frame), "Columns": frame.shape[1], "Path": str(path)})

written_df = pd.DataFrame(written).sort_values("Table").reset_index(drop=True)
written_df.to_csv(PATHS.reports / "interim_outputs.csv", index=False)
written_df


## 10. Final readiness checks

In [ ]:
required_interim = [
    "teams.parquet",
    "seasons.parquet",
    "seeds.parquet",
    "games_compact_canonical.parquet",
    "tournament_targets.parquet",
    "games_detailed_team_long.parquet",
    "submission_matchups.parquet",
]
missing = [name for name in required_interim if not (PATHS.interim / name).exists()]
assert not missing, f"Missing interim outputs: {missing}"

summary = {
    "raw_tables_loaded": len(tables),
    "raw_rows_total": int(sum(len(frame) for frame in tables.values())),
    "canonical_compact_games": int(len(games_compact)),
    "detailed_team_game_rows": int(len(games_detailed_team_long)),
    "historical_tournament_targets": int(len(tournament_targets)),
    "submission_matchups": int(len(submission_matchups)),
    "blocking_audit_failures": int(len(errors)),
}
print(json.dumps(summary, indent=2))
print("\nDATA PREPARATION COMPLETE — ready for leakage-safe feature engineering.")


## Stop here

The next notebook should build **fold-specific, pre-tournament team-season snapshots**. Do not aggregate the entire dataset once and then cross-validate; that can leak future seasons, tournament outcomes, or post-cutoff rankings into validation folds.